# Day 17 — Agent / ReAct：从“生成 token”到“Reason → Act → Observe”

> Week 3 主线：LLM Inference → Agent → Graph × LLM

今天的唯一主论文：

**ReAct: Synergizing Reasoning and Acting in Language Models**  
Yao et al., ICLR 2023

论文链接：
- arXiv: https://arxiv.org/abs/2210.03629
- PDF: https://arxiv.org/pdf/2210.03629
- OpenReview: https://openreview.net/forum?id=WE_vluYUL-X

今天不要开 LangChain / LlamaIndex / AutoGen / CrewAI。  
目标不是学框架，而是建立 Agent 的最小计算图：

```text
Observation
    ↓
Thought
    ↓
Action
    ↓
Environment / Tool
    ↓
Observation
    ↓
Thought
    ↓
...
```

Day 16 关注：
> LLM 如何更高效地产生 token？

Day 17 关注：
> LLM 如何把 token 变成“行动”，并利用环境反馈继续决策？

## 今天要读什么

按下面顺序读，不需要从第一页线性啃到最后：

1. Abstract
2. Introduction
3. Figure / trajectory examples
4. Method / formulation
5. 关键实验
6. Discussion / limitations

今天读论文时始终带着 6 个问题：

1. 为什么 reasoning-only 不够？
2. 为什么 acting-only 不够？
3. Thought / Action / Observation 各自是什么？
4. ReAct 到底新增了什么 loop？
5. Environment / tool 在系统里扮演什么角色？
6. ReAct 和普通 chain-of-thought 的本质区别是什么？

---
# 1. 先建立最小 Agent 直觉

普通 LLM：

```text
Prompt
  ↓
LLM
  ↓
Answer
```

Agent / ReAct：

```text
Task
 ↓
LLM
 ↓
Thought
 ↓
Action
 ↓
Tool / Environment
 ↓
Observation
 ↓
LLM
 ↓
Next Thought / Action
 ↓
...
 ↓
Final Answer
```

关键变化不是“模型参数突然变了”，而是：

> **LLM 被放进了一个外部控制循环里。**

这和 Day 16 的 autoregressive token loop 不一样：

```text
token loop:
x_t → x_{t+1} → x_{t+2}

agent loop:
observation → thought → action → observation → ...
```

### 必须回答 1

普通 LLM 和 Agent 最大的结构区别是什么？

请不要回答“Agent 更智能”。

回答：

> 普通 LLM 通常是 input → generation → output 的单次生成过程；Agent 则将 LLM 作为决策核心，嵌入 reasoning → action → environment/tool → observation → reasoning → ... 的外部交互循环，并维护交互 state，直到满足终止条件后输出 Final Answer

---
# 2. Reasoning-only 为什么不够？

Reasoning-only 可以理解成：

```text
Question
 ↓
Thought
 ↓
Thought
 ↓
Thought
 ↓
Answer
```

问题是：

- 可能基于错误记忆继续推理；
- 无法主动获取缺失事实；
- 无法修改环境；
- 一旦前提错了，后面的 reasoning 可能越走越偏。

所以 reasoning-only 的局限是：

$$
\boxed{\text{会想，但不一定能验证 / 行动}}
$$

### 必须回答 2

Reasoning-only 的 error propagation，和 ReAct 中可能出现的 error propagation 有什么区别？ReAct 是否彻底解决了这个问题？

回答：Reasoning-only 的 error propagation 是指某一步出现错误事实或错误推理后，后续 reasoning 继续把这个错误结果作为前提，从而使错误沿 reasoning chain 不断传播。

ReAct 并没有彻底解决 error propagation。 外部 Observation 可以为模型提供 grounding，使 Agent 有机会发现并修正之前的错误；但 ReAct 同时引入了新的错误来源，例如错误的 Action、错误的 tool selection、噪声或错误的 Observation，以及模型对 Observation 的错误理解。这些错误同样可能沿后续 Agent trajectory 继续传播。

---
# 3. Acting-only 为什么也不够？

Acting-only 可以理解成：

```text
Observation
 ↓
Action
 ↓
Observation
 ↓
Action
```

如果没有中间 reasoning：

- 容易做局部、短视动作；
- 不容易维护长期计划；
- 对多步任务缺乏显式中间状态；
- 很难解释“为什么下一步要这么做”。

所以 acting-only 的局限是：

$$
\boxed{\text{会做，但不一定会规划}}
$$

### 必须回答 3

为什么 acting-only 也不够？

请从 planning、long-horizon task、intermediate reasoning state 三个角度解释。

回答：
Acting-only 的问题在于它虽然能够与环境交互，但缺乏显式 reasoning 来组织这些 action。

1.Planning： 如果没有 reasoning，Agent 往往只能根据当前 observation 做局部反应，缺少“先做什么、后做什么”的整体计划，容易变得短视。

2.Long-horizon task： 对需要很多步骤才能完成的任务，Agent 需要持续追踪目标、子目标以及已经完成的步骤。只依赖 Observation → Action 很容易在长流程中偏离最初目标。

3.ntermediate reasoning state： reasoning 可以显式保存“当前已经知道什么、还缺什么、为什么下一步要执行这个 action”。如果缺少这些中间 reasoning state，那么行为过程更难解释，也更难在出现异常 observation 时调整计划。

因此，acting-only 可以理解成“会做，但不一定会规划为什么这样做”；ReAct 则通过 Thought 来指导 Action，使长期决策更加有结构。

---
# 4. ReAct 的核心：Thought + Action + Observation

ReAct 把 trajectory 写成类似：

```text
Thought 1: 我需要先查 A
Action 1: Search[A]
Observation 1: 得到结果 ...

Thought 2: 这个结果说明 ...
Action 2: Lookup[B]
Observation 2: ...

Thought 3: 信息够了
Action 3: Finish[answer]
```

这里：

- **Thought**：内部推理 / 计划
- **Action**：发给环境或工具的操作
- **Observation**：环境返回的新信息

最重要的是：

> Observation 会进入下一轮上下文，改变后续决策。

### 必须回答 4

Thought / Action / Observation 三者分别是什么？

请特别解释：
> 为什么 Observation 不是“模型自己想出来的下一句话”？

Thought： Agent 根据当前任务和已有 trajectory 生成的内部 reasoning / planning，用来判断当前知道什么、缺什么信息以及下一步应该做什么。

Action： Agent 根据 Thought 对外部环境或工具发出的操作，例如 Search[...]、调用 API、运行代码或执行环境中的某个动作。

Observation： Action 被外部工具或环境执行后返回给 Agent 的结果。Observation 会进入下一轮上下文，并影响后续 Thought 和 Action

---
# 5. 一个极简 ReAct 控制流实验

下面不调用真实 LLM，只模拟控制流。

In [1]:
knowledge = {
    "france_capital": "Paris",
    "paris_country": "France",
}

def tool_call(action, arg):
    if action == "lookup":
        return knowledge.get(arg, "NOT_FOUND")
    if action == "calculator":
        return str(eval(arg))
    return "UNKNOWN_ACTION"

trajectory = []

steps = [
    ("Thought", "我需要先查法国首都"),
    ("Action", ("lookup", "france_capital")),
    ("Thought", "已经得到首都，可以回答"),
    ("Final", "Paris"),
]

for kind, payload in steps:
    if kind == "Action":
        action, arg = payload
        obs = tool_call(action, arg)
        trajectory.append(("Action", payload))
        trajectory.append(("Observation", obs))
    else:
        trajectory.append((kind, payload))

for item in trajectory:
    print(item)

('Thought', '我需要先查法国首都')
('Action', ('lookup', 'france_capital'))
('Observation', 'Paris')
('Thought', '已经得到首都，可以回答')
('Final', 'Paris')


### 必须回答 5

这段 toy code 想说明什么？

不是让你背代码，而是回答：

> Agent 的核心是不是一个“LLM + tool + state + loop”结构？  YES

---
# 6. ReAct 为什么比普通 CoT 多一步？

普通 Chain-of-Thought：

```text
Question
 ↓
Thought
 ↓
Thought
 ↓
Answer
```

ReAct：

```text
Question
 ↓
Thought
 ↓
Action
 ↓
Observation
 ↓
Thought
 ↓
...
```

所以：

$$
\boxed{\text{CoT 主要在 model-internal reasoning}}
$$

而：

$$
\boxed{\text{ReAct 把 reasoning 和 external interaction 交织起来}}
$$

### 必须回答 6

ReAct 和普通 CoT 的本质区别是什么？

不要只回答：
> ReAct 多了 Action。

要解释：
> Action 带回来的 Observation 如何改变后续 reasoning。

回答：CoT 主要是在模型内部连续生成 reasoning chain，后续 reasoning 主要依赖原始问题和前面的 reasoning；ReAct 则把 reasoning 与对外部环境的 Action 交错起来，Action 得到的真实 Observation 会重新进入模型的 context，从而改变下一轮 Thought 和 Action。因此 ReAct 形成了一个能够被外部信息动态修正的交互闭环。

---
# 7. Agent 里的 state 是什么？

Agent 不只是“每轮重新问 LLM”。

它通常需要维护：

```text
task
+
previous thoughts
+
actions
+
observations
+
tool results
+
possibly memory
```

这些共同构成当前决策的 state / trajectory。

可以把 trajectory 想成：

$$
\tau_t=(o_1,a_1,o_2,a_2,\ldots,o_t)
$$

下一步行为：

$$
a_t \sim \pi_\theta(\cdot \mid \tau_t)
$$

今天不要求强化学习推导，只要理解：

> **Agent 的下一步依赖整个交互历史，而不是只看原始 prompt。**

### 必须回答 7

为什么 trajectory / state 对 Agent 很重要？

如果每次 tool call 后都忘记前面的 observation，会发生什么？

回答：Agent 的下一步决策不仅依赖原始 task，还依赖此前已经发生的交互历史。Trajectory 保存 Thought、Action、Observation 等历史信息；这些信息构成或参与构成当前 state，使 Agent 知道自己已经做过什么、获得了什么信息以及下一步还需要做什么。如果每次 tool call 后都忘记之前的 Observation，Agent 就无法利用前面的工具结果，可能重复执行相同 Action、丢失任务进度，甚至无法完成需要多步交互的 long-horizon task。

---
# 8. Tool use：Agent 什么时候该用工具？

一个重要原则：

> 工具不是“越多越好”，而是当模型自身信息或能力不足时，外部工具提供新的可靠 observation。

常见 tool：

```text
calculator
search
database lookup
code execution
graph lookup
file retrieval
API call
```

今天不要学任何 framework，只理解 tool interface：

```python
result = tool(name, arguments)
```

然后：

```text
result → Observation → 回到模型
```

### 必须回答 8

为什么 tool output 应该作为 Observation 返回给模型，而不是直接当 Final Answer？

回答：Tool output 通常只是 Agent 获取到的一份外部信息，而不一定直接等于用户问题的答案。它应该作为 Observation 返回给 Agent，使 LLM 能够对结果进行解释、验证、组合，并判断是否还需要继续调用其他工具。只有当 Agent 判断已有信息足以完成任务时，才生成 Final Answer。

---
# 9. 论文实验看什么

今天不要陷入所有 benchmark 细节。

重点看：

1. ReAct 在 knowledge-intensive / interactive tasks 上是否优于只 reasoning 或只 acting？
2. 哪些任务最能体现“reasoning + acting”互补？
3. 失败案例是什么？
4. ReAct 的轨迹为什么更可解释？
5. 它是否依赖 prompt examples / demonstrations？

看实验时始终问：

$$
\boxed{\text{到底是哪一个 component 带来了收益？}}
$$

这和 Week 2 做 ablation 的思路完全一致。

### 必须回答 9

如果 ReAct 比 CoT 更好，你怎么证明收益真的是来自 “Action/Observation loop”，而不是因为 prompt 更长？

请给一个最基本的 controlled comparison 思路。

回答：为了证明 ReAct 的收益来自 Action/Observation loop，而不是 prompt 更长，应尽量控制模型、任务、示例数量、prompt 长度等因素不变，只改变是否加入外部 Action 和 Observation。比如比较 CoT 与 ReAct：两者都保留 reasoning demonstrations，但 ReAct 额外允许工具调用并将 Observation 返回给下一轮 reasoning。若 ReAct 仍显著更好，才更有证据说明收益来自 external interaction loop，而不是单纯更多 token 或更长 prompt。

---
# 10. ReAct 的局限

今天至少要意识到：

- tool / environment 可能返回错误信息；
- 模型可能选择错误 action；
- reasoning trajectory 可能很长，增加 latency / token cost；
- tool call 本身有 latency；
- observation 可能污染上下文；
- agent loop 可能不停止；
- 多步错误可能累计。

所以 Agent 并不是：

$$
\text{LLM} + \text{tool} = \text{自动正确}
$$

而是引入了新的系统与决策问题。

### 必须回答 10

Agent 相比普通单次 LLM inference，新引入了哪些 cost / failure modes？

至少回答三个。

回答：Agent 相比普通单次 LLM inference，引入了更多系统成本和 failure modes。
成本方面，一个 Agent task 可能需要多轮 LLM inference、多次 tool call，并不断把 trajectory / observation 加入 context，因此会增加 token cost、LLM inference latency、tool latency 和 context/memory 开销。
Failure modes 方面，Agent 可能选择错误的 tool/action、得到错误或噪声 Observation、错误理解 Observation，也可能让早期错误沿 trajectory 传播，甚至出现重复调用工具或 agent loop 无法终止的问题。

---
# 11. 和 Day 16 串起来

Day 16：

```text
一个 request
↓
Prefill / Decode
↓
KV cache
↓
latency / throughput
```

Day 17：

```text
一个 Agent task
↓
LLM request
↓
Action
↓
Tool call
↓
Observation
↓
新的 LLM request
↓
...
```

所以一个 Agent task 往往不是一次 LLM inference，而是：

$$
\boxed{\text{多轮 LLM inference + tool calls + state updates}}
$$

这会自然带来：

- 更长 latency
- 更多 token cost
- 更动态的 serving workload
- 更复杂的 scheduling / memory 问题

这也是为什么 Agent 和 LLM Systems 最终会重新连起来。

### 必须回答 11

为什么 Agent workload 比普通 chat completion 更“系统化”和更动态？

请结合：
- 多轮 LLM call
- tool latency
- observation length
- variable number of steps

回答：普通 chat completion 通常可以近似看成一次相对确定的 LLM generation，而 Agent task 的执行路径是动态的。
Agent 可能进行多轮 LLM calls，每轮可能选择不同 tool；不同 tool 的 latency 不同；返回的 Observation 长度不同；而 Observation 又会影响下一轮决策，因此整个 Agent 到底执行多少 steps、产生多少 tokens、调用哪些 tools、什么时候结束，都可能事先未知。这使 Agent workload 在 latency、compute、memory 和 scheduling 上更加动态和复杂。

---
# 12. Day 17 最终复盘

请不翻答案，直接回答：

1. Agent 相比普通 LLM 多了什么结构？
2. Reasoning-only 为什么不够？
3. Acting-only 为什么不够？
4. Thought / Action / Observation 分别是什么？
5. ReAct 和 CoT 的本质区别是什么？
6. 为什么 Observation 会改变下一步决策？
7. trajectory / state 为什么必要？
8. tool use 的核心价值是什么？
9. 怎么验证 ReAct 的收益来自 action-observation loop？
10. Agent 新引入了哪些 latency / cost / failure mode？
11. 用一句话总结 ReAct。

建议一句话：

> ReAct 把 LLM 从“只在内部生成 reasoning”变成了“在 reasoning、action 与 environment observation 之间循环决策”的 agentic computation pattern。

# Day 17 完成标准

今天不要求：

- 实现真实 LLM Agent；
- 安装 LangChain；
- 写复杂 tool router；
- 学多 Agent；
- 学 memory framework；
- 学 RL agent theory。

今天要求：

```text
看懂 ReAct 论文主线
↓
理解 Thought / Action / Observation
↓
理解 Agent = LLM + Environment + Loop + State
↓
知道为什么 reasoning 和 acting 要交织
↓
能指出 Agent 新引入的 systems / reliability 问题
```

Day 18：

> 自己手写最小 Agent：Plan → Tool → Observation → Next Action loop